# STAT 481 — MLR Analysis of Cereal Ratings
**Author:** Bhavisha Ahuja  
**Course:** STAT 481: Applied Regression | UIC | Fall 2025

Predicts Consumer Reports cereal ratings from nutritional data using multiple linear regression. Includes full assumption checking, multicollinearity testing, Box-Cox transformation exploration, and backward elimination variable selection.

**Final model:** `rating = 61.08 − 0.0559(sodium) + 2.7525(fiber) − 2.1904(sugars)` — R² ≈ 0.92

## 1. Setup & Data Loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.stats.diagnostic import het_breuschpagan
from statsmodels.stats.stattools import durbin_watson
from scipy import stats

# Load and clean data
df = pd.read_csv("CerealsRating.csv")

cols = ["protein", "sodium", "fiber", "sugars", "vitamins", "weight", "cups", "rating"]
df = df[cols]
df = df.replace(-1, np.nan).dropna()
df.columns = ["x1", "x2", "x3", "x4", "x5", "x6", "x7", "y"]

print("Dataset shape:", df.shape)
print(df.head())

## 2. Descriptive Statistics

In [ ]:
print("Summary Statistics:")
df.describe().round(2)

## 3. Visual Exploration

In [ ]:
# Histograms
df.hist(figsize=(12, 8), bins=15)
plt.suptitle("Histograms of All Variables")
plt.tight_layout()
plt.show()

In [ ]:
# Boxplots
plt.figure(figsize=(12, 6))
sns.boxplot(data=df)
plt.title("Boxplots of Predictors and Response")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Correlation matrix
plt.figure(figsize=(9, 7))
sns.heatmap(df.corr(), annot=True, cmap="coolwarm", center=0)
plt.title("Correlation Matrix")
plt.tight_layout()
plt.show()

## 4. Full Multiple Linear Regression Model

In [ ]:
# Fit full model
full_model = smf.ols("y ~ x1 + x2 + x3 + x4 + x5 + x6 + x7", data=df).fit()
print(full_model.summary())

In [ ]:
# ANOVA table
anova_table = sm.stats.anova_lm(full_model, typ=2)
print("ANOVA Table:")
print(anova_table)

## 5. Multicollinearity Check (VIF)

In [ ]:
X = df[["x1", "x2", "x3", "x4", "x5", "x6", "x7"]]
vif_df = pd.DataFrame()
vif_df["Variable"] = X.columns
vif_df["VIF"] = [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]
print("VIF Values:")
print(vif_df)

## 6. Model Assumption Checks — Full Model

In [ ]:
resid = full_model.resid
fitted = full_model.fittedvalues

# Residuals vs Fitted
plt.figure(figsize=(10, 6))
plt.scatter(fitted, resid, alpha=0.6)
plt.axhline(0, color='red', linestyle='--')
plt.xlabel("Fitted Values")
plt.ylabel("Residuals")
plt.title("Residuals vs Fitted Values — Full Model")
plt.show()

# Q-Q plot
sm.qqplot(resid, line='45')
plt.title("Q-Q Plot of Residuals — Full Model")
plt.show()

# Normality test
sw_stat, sw_pval = stats.shapiro(resid)
print(f"Shapiro-Wilk Test: W={sw_stat:.4f}, p={sw_pval:.4f}")

# Heteroscedasticity
X_const = sm.add_constant(X)
bp_stat, bp_pval, _, _ = het_breuschpagan(resid, X_const)
print(f"Breusch-Pagan Test: stat={bp_stat:.4f}, p={bp_pval:.4f}")

# Autocorrelation
dw_stat = durbin_watson(resid)
print(f"Durbin-Watson: {dw_stat:.4f}")

## 7. Box-Cox Transformation Exploration

In [ ]:
y_pos = df["y"] - df["y"].min() + 1
y_transformed, lam = stats.boxcox(y_pos)
print(f"Box-Cox lambda: {lam:.4f}")
# Lambda ≈ -0.41 — transformation did not meaningfully improve normality
# or change significant predictors; proceeded with original scale for interpretability

## 8. Variable Selection — Backward Elimination (α = 0.10)

In [ ]:
def backward_elimination(data, response, alpha=0.10):
    predictors = [c for c in data.columns if c != response]
    step = 1
    while True:
        formula = f"{response} ~ " + " + ".join(predictors)
        model = smf.ols(formula, data=data).fit()
        pvals = model.pvalues[1:]
        max_p = pvals.max()
        print(f"\nStep {step}: max p-value = {max_p:.4f} for {pvals.idxmax()}")
        if max_p > alpha:
            remove_var = pvals.idxmax()
            print(f"  Removing {remove_var}")
            predictors.remove(remove_var)
            step += 1
        else:
            print("  All remaining variables significant. Stopping.")
            break
    final_formula = f"{response} ~ " + " + ".join(predictors)
    final_model = smf.ols(final_formula, data=data).fit()
    return final_model, predictors

final_model, final_vars = backward_elimination(df, "y")
print("\n" + "="*60)
print("FINAL MODEL")
print("="*60)
print(f"Variables retained: {final_vars}")
print(final_model.summary())

## 9. Final Model Assumption Checks

In [ ]:
resid_final = final_model.resid
fitted_final = final_model.fittedvalues

# Residuals vs Fitted
plt.figure(figsize=(10, 6))
plt.scatter(fitted_final, resid_final, alpha=0.6)
plt.axhline(0, color='red', linestyle='--')
plt.title("Final Model: Residuals vs Fitted")
plt.xlabel("Fitted Values")
plt.ylabel("Residuals")
plt.show()

# Q-Q plot
sm.qqplot(resid_final, line='45')
plt.title("Final Model: Q-Q Plot")
plt.show()

# Diagnostic tests
sw_f, sw_p_f = stats.shapiro(resid_final)
print(f"Shapiro-Wilk: W={sw_f:.4f}, p={sw_p_f:.4f}")

X_final = df[final_vars]
X_final_const = sm.add_constant(X_final)
bp_f, bp_p_f, _, _ = het_breuschpagan(resid_final, X_final_const)
print(f"Breusch-Pagan: stat={bp_f:.4f}, p={bp_p_f:.4f}")

dw_f = durbin_watson(resid_final)
print(f"Durbin-Watson: {dw_f:.4f}")

## 10. Model Comparison

In [ ]:
print("="*60)
print("MODEL COMPARISON")
print("="*60)
print(f"Full Model  (7 predictors): R²={full_model.rsquared:.3f}, Adj R²={full_model.rsquared_adj:.3f}")
print(f"Final Model (3 predictors): R²={final_model.rsquared:.3f}, Adj R²={final_model.rsquared_adj:.3f}")
print(f"\nFinal model equation:")
print(f"rating = {final_model.params['Intercept']:.4f} + {final_model.params['x2']:.4f}(sodium) + {final_model.params['x3']:.4f}(fiber) + {final_model.params['x4']:.4f}(sugars)")